# FT-Transformer — Probability targets (`y_24h`, `y_72h`)
Instead of treating all features as a flat vector
(like an MLP does), it projects **each feature into its own learned embedding token**, then
runs standard Transformer self-attention across all feature tokens.

This means the model learns interactions like
*"shallow depth AND large magnitude AND strike-slip → aftershock"* inside a single
attention head — without us manually engineering those products. This is the key
advantage over the MLP we already trained.

**Architecture choices for this dataset:**
- `d_token = 32` — embedding dimension per feature (small; we have 23k rows not 1M)
- `n_heads = 4` — attention heads
- `n_layers = 3` — transformer blocks
- `dropout = 0.15` — applied inside each block
- Two separate output heads share all transformer layers (multi-task: y_24h + y_72h jointly)
- Temperature scaling for calibration (same as MLP notebook)

In [1]:
from __future__ import annotations

import copy
import math
import random
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score
from torch.utils.data import DataLoader, TensorDataset

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find repo root containing /src")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation.metrics import evaluate_binary_probabilities
from src.models.feature_sets import BASE_TABULAR_FEATURES, GCMT_FEATURES, QUALITY_FEATURES
from src.models.input_layer import InputConfig, load_modeling_splits, prepare_tabular_inputs
from src.utils.paths import METRICS_DIR

DATASET_NAME = "earthquake_aftershock_v2_gcmt"
MODEL_NAME   = "ftt_v1"
TARGETS      = ["y_24h", "y_72h"]
DEVICE       = "cuda" if torch.cuda.is_available() else "cpu"

def seed_everything(seed: int = 42) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)

seed_everything(42)
print(f"Project root : {PROJECT_ROOT}")
print(f"Device       : {DEVICE}")
print(f"PyTorch      : {torch.__version__}")

Project root : /Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10
Device       : cpu
PyTorch      : 2.10.0


In [2]:
splits = load_modeling_splits(dataset_name=DATASET_NAME)

for split_name, df in splits.items():
    print(f"{split_name:5s}  {len(df):>5} rows  |  "
          f"years {df['trigger_year'].min()}–{df['trigger_year'].max()}  |  "
          f"y_24h pos: {df['y_24h'].mean():.3f}")

train  23031 rows  |  years 2010–2022  |  y_24h pos: 0.443
val     1744 rows  |  years 2023–2023  |  y_24h pos: 0.478
test    3513 rows  |  years 2024–2025  |  y_24h pos: 0.469


In [3]:
def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    has_gcmt = (
        df["has_gcmt"].fillna(False)
        if "has_gcmt" in df.columns
        else pd.Series(False, index=df.index)
    )

    # Depth regime
    df["depth_shallow"]      = (df["trigger_depth_km"] < 70).astype(float)
    df["depth_intermediate"] = df["trigger_depth_km"].between(70, 300).astype(float)
    df["depth_deep"]         = (df["trigger_depth_km"] > 300).astype(float)

    # Magnitude transforms
    df["log_magnitude"]    = np.log10(df["trigger_magnitude"].clip(lower=1e-6))
    df["mag_depth_ratio"]  = df["trigger_magnitude"] / (df["trigger_depth_km"] + 1)

    # Log scalar moment (GCMT-only; NaN otherwise — handled by median impute)
    df["log_scalar_moment"] = np.where(
        has_gcmt & df["gcmt_scalar_moment"].notna(),
        np.log10(df["gcmt_scalar_moment"].clip(lower=1e-10)),
        np.nan,
    )

    # Temporal cyclical encodings
    df["sin_month"]      = np.sin(2 * np.pi * df["trigger_month"]     / 12)
    df["cos_month"]      = np.cos(2 * np.pi * df["trigger_month"]     / 12)
    df["sin_hour"]       = np.sin(2 * np.pi * df["trigger_hour"]      / 24)
    df["cos_hour"]       = np.cos(2 * np.pi * df["trigger_hour"]      / 24)
    df["sin_dayofyear"]  = np.sin(2 * np.pi * df["trigger_dayofyear"] / 365)
    df["cos_dayofyear"]  = np.cos(2 * np.pi * df["trigger_dayofyear"] / 365)

    # Spatial: circum-Pacific belt
    df["ring_of_fire"] = (
        (np.abs(df["trigger_longitude"]) > 130)
        & df["trigger_latitude"].between(-60, 60)
    ).astype(float)

    # Tectonic regime from GCMT rake angle
    def _rake_regimes(rake):
        if pd.isna(rake): return np.nan, np.nan, np.nan
        r = rake % 360
        ss = min(abs(r), abs(r - 180), abs(r - 360))
        return float(ss < 45), float(45 <= r <= 135), float(225 <= r <= 315)

    regimes = df["rake"].apply(
        lambda r: _rake_regimes(r) if pd.notna(r) else (np.nan, np.nan, np.nan)
    )
    df["is_strike_slip"] = regimes.apply(lambda x: x[0]).where(has_gcmt)
    df["is_reverse"]     = regimes.apply(lambda x: x[1]).where(has_gcmt)
    df["is_normal"]      = regimes.apply(lambda x: x[2]).where(has_gcmt)

    # Rupture geometry
    df["sin_dip"] = np.sin(np.radians(df["dip"].where(has_gcmt)))
    e1 = df["gcmt_eig1"].where(has_gcmt)
    e2 = df["gcmt_eig2"].where(has_gcmt)
    e3 = df["gcmt_eig3"].where(has_gcmt)
    denom = (e1.abs() + e3.abs()).replace(0, np.nan)
    df["clvd_fraction"]       = (2 * e2.abs() / denom).where(has_gcmt)
    df["centroid_depth_diff"] = (df["gcmt_depth_km"] - df["trigger_depth_km"]).where(has_gcmt)
    df["log_half_duration"]   = np.log1p(df["gcmt_half_duration_sec"].where(has_gcmt))
    df["mag_diff_abs"]        = df["gcmt_mag_diff"].abs().where(has_gcmt)

    # Moment exponent (order-of-magnitude of seismic moment; centered at 24)
    df["moment_exponent_centered"] = (
        df["gcmt_moment_exponent"].where(has_gcmt) - 24.0
    )

    # Seismicity rate features
    prior_24h = df["prior_global_event_count_24h"]
    prior_7d  = df["prior_global_event_count_7d"]
    df["log_prior_24h"]           = np.log1p(prior_24h)
    df["log_prior_7d"]            = np.log1p(prior_7d)
    df["seismicity_acceleration"] = prior_24h / (prior_7d / 7.0).replace(0, np.nan)

    # Interaction features
    df["mag_x_shallow"]    = df["trigger_magnitude"] * df["depth_shallow"]
    df["mag_x_log_prior"]  = df["trigger_magnitude"] * np.log1p(prior_24h)

    return df


ENGINEERED_FEATURES = [
    "depth_shallow", "depth_intermediate", "depth_deep",
    "log_magnitude", "mag_depth_ratio", "log_scalar_moment",
    "sin_month", "cos_month", "sin_hour", "cos_hour",
    "sin_dayofyear", "cos_dayofyear", "ring_of_fire",
    "is_strike_slip", "is_reverse", "is_normal",
    "sin_dip", "clvd_fraction", "centroid_depth_diff",
    "log_half_duration", "mag_diff_abs",
    "moment_exponent_centered",
    "log_prior_24h", "log_prior_7d", "seismicity_acceleration",
    "mag_x_shallow", "mag_x_log_prior",
]

_seen: set[str] = set()
FULL_FEATURE_SET: list[str] = []
for f in BASE_TABULAR_FEATURES + QUALITY_FEATURES + GCMT_FEATURES + ENGINEERED_FEATURES:
    if f not in _seen:
        FULL_FEATURE_SET.append(f)
        _seen.add(f)

splits_eng = {name: engineer_features(df) for name, df in splits.items()}
print(f"Total features requested: {len(FULL_FEATURE_SET)}")
print("Feature engineering complete.")

Total features requested: 83
Feature engineering complete.


In [4]:
prepared: dict[str, object] = {}

for target in TARGETS:
    config = InputConfig(
        feature_cols=FULL_FEATURE_SET,
        target_col=target,
        missing_strategy="median",   # FT-Transformer uses standard imputation
        scale=True,                  # standardise to zero-mean, unit-variance
        allow_missing_optional=True,
        drop_rows_with_missing_target=True,
    )
    inputs = prepare_tabular_inputs(config=config, splits=splits_eng)
    prepared[target] = inputs
    print(f"{target} — X_train: {inputs.X_train.shape}, "
          f"features resolved: {len(inputs.feature_cols)}")

y_24h — X_train: (23031, 83), features resolved: 83
y_72h — X_train: (23031, 83), features resolved: 83


In [5]:
class FeatureTokenizer(nn.Module):
    """Projects each scalar feature to a d_token-dim embedding vector."""
    def __init__(self, n_features: int, d_token: int) -> None:
        super().__init__()
        self.weight = nn.Parameter(torch.randn(n_features, d_token) * 0.02)
        self.bias   = nn.Parameter(torch.zeros(n_features, d_token))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: (B, F)  →  (B, F, d_token)
        return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)


class FTTransformerBlock(nn.Module):
    def __init__(self, d_token: int, n_heads: int, ffn_factor: float = 2.0,
                 dropout: float = 0.1) -> None:
        super().__init__()
        self.norm1 = nn.LayerNorm(d_token)
        self.attn  = nn.MultiheadAttention(d_token, n_heads, dropout=dropout,
                                           batch_first=True)
        self.norm2 = nn.LayerNorm(d_token)
        d_ffn = int(d_token * ffn_factor)
        self.ffn = nn.Sequential(
            nn.Linear(d_token, d_ffn), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d_ffn, d_token), nn.Dropout(dropout),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-norm: normalise BEFORE attention and FFN (more stable)
        h = self.norm1(x)
        h, _ = self.attn(h, h, h)
        x = x + h
        x = x + self.ffn(self.norm2(x))
        return x


class FTTransformer(nn.Module):
    """
    Feature Tokenizer + Transformer for tabular binary classification.
    Gorishniy et al., NeurIPS 2021 (https://arxiv.org/abs/2106.11959).
    """
    def __init__(
        self,
        n_features: int,
        d_token: int    = 32,
        n_heads: int    = 4,
        n_layers: int   = 3,
        ffn_factor: float = 2.0,
        dropout: float  = 0.15,
    ) -> None:
        super().__init__()
        self.tokenizer = FeatureTokenizer(n_features, d_token)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        self.blocks = nn.ModuleList([
            FTTransformerBlock(d_token, n_heads, ffn_factor, dropout)
            for _ in range(n_layers)
        ])
        self.norm = nn.LayerNorm(d_token)
        # Separate heads — share all transformer layers (multi-task learning)
        self.head_24h = nn.Linear(d_token, 1)
        self.head_72h = nn.Linear(d_token, 1)

    def forward(self, x: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        B = x.shape[0]
        tokens = self.tokenizer(x)                         # (B, F, d)
        cls    = self.cls_token.expand(B, -1, -1)          # (B, 1, d)
        tokens = torch.cat([cls, tokens], dim=1)            # (B, F+1, d)
        for block in self.blocks:
            tokens = block(tokens)
        cls_out = self.norm(tokens[:, 0])                   # (B, d) — CLS position
        p24 = torch.sigmoid(self.head_24h(cls_out)).squeeze(1)
        p72 = torch.sigmoid(self.head_72h(cls_out)).squeeze(1)
        return p24, p72


# Quick sanity check
with torch.no_grad():
    _n = len(prepared["y_24h"].feature_cols)
    _m = FTTransformer(_n)
    _x = torch.randn(4, _n)
    _p24, _p72 = _m(_x)
    assert _p24.shape == (4,) and _p72.shape == (4,)
    n_params = sum(p.numel() for p in _m.parameters())
    print(f"Architecture OK — {_n} features, {n_params:,} parameters")
    del _m, _x, _p24, _p72

Architecture OK — 83 features, 31,106 parameters


In [6]:
def make_loaders(
    X_train: pd.DataFrame,
    y_train: pd.Series,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    batch_size: int = 512,
) -> tuple[DataLoader, DataLoader]:
    Xtr = torch.tensor(X_train.values.astype(np.float32))
    ytr = torch.tensor(y_train.values.astype(np.float32))
    Xva = torch.tensor(X_val.values.astype(np.float32))
    yva = torch.tensor(y_val.values.astype(np.float32))
    train_ds = TensorDataset(Xtr, ytr)
    val_ds   = TensorDataset(Xva, yva)
    return (
        DataLoader(train_ds, batch_size=batch_size, shuffle=True,  drop_last=True),
        DataLoader(val_ds,   batch_size=batch_size, shuffle=False),
    )


class TemperatureScaler(nn.Module):
    """Single learnable temperature parameter for post-hoc calibration."""
    def __init__(self) -> None:
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1))

    def forward(self, logits: torch.Tensor) -> torch.Tensor:
        return torch.sigmoid(logits / self.temperature.clamp(min=0.01))

    def fit(self, logits: torch.Tensor, labels: torch.Tensor,
            lr: float = 0.05, max_steps: int = 200) -> None:
        opt = torch.optim.LBFGS([self.temperature], lr=lr, max_iter=max_steps)
        def closure():
            opt.zero_grad()
            loss = nn.BCELoss()(self.forward(logits), labels)
            loss.backward()
            return loss
        opt.step(closure)

In [7]:
FTT_PARAMS = dict(
    d_token    = 32,
    n_heads    = 4,
    n_layers   = 3,
    ffn_factor = 2.0,
    dropout    = 0.15,
)

TRAIN_PARAMS = dict(
    lr           = 1e-4,
    weight_decay = 1e-4,
    batch_size   = 512,
    max_epochs   = 200,
    patience     = 25,
)

trained_models: dict[str, dict] = {}

for target in TARGETS:
    print(f"\n===== Training {target} =====")
    inp = prepared[target]
    n_feats = inp.X_train.shape[1]

    train_loader, val_loader = make_loaders(
        inp.X_train, inp.y_train,
        inp.X_val,   inp.y_val,
        batch_size=TRAIN_PARAMS["batch_size"],
    )

    model = FTTransformer(n_feats, **FTT_PARAMS).to(DEVICE)
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=TRAIN_PARAMS["lr"],
        weight_decay=TRAIN_PARAMS["weight_decay"],
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=TRAIN_PARAMS["max_epochs"], eta_min=1e-6
    )
    criterion = nn.BCELoss()

    best_val_auc = 0.0
    best_state   = None
    no_improve   = 0

    for epoch in range(1, TRAIN_PARAMS["max_epochs"] + 1):
        # ── train
        model.train()
        epoch_loss = 0.0
        for Xb, yb in train_loader:
            Xb, yb = Xb.to(DEVICE), yb.to(DEVICE)
            p24, p72 = model(Xb)
            # Use the appropriate head depending on which target we're training
            pred = p24 if target == "y_24h" else p72
            loss = criterion(pred, yb)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
            epoch_loss += loss.item()
        scheduler.step()

        # ── validate
        model.eval()
        val_probs, val_labels = [], []
        with torch.no_grad():
            for Xb, yb in val_loader:
                p24, p72 = model(Xb.to(DEVICE))
                pred = p24 if target == "y_24h" else p72
                val_probs.append(pred.cpu()); val_labels.append(yb)
        val_probs  = torch.cat(val_probs).numpy()
        val_labels = torch.cat(val_labels).numpy()
        val_auc    = roc_auc_score(val_labels, val_probs)

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state   = copy.deepcopy(model.state_dict())
            no_improve   = 0
        else:
            no_improve += 1

        if epoch % 25 == 0 or epoch == 1:
            print(f"  Epoch {epoch:3d} | val_auc={val_auc:.4f} | "
                  f"best={best_val_auc:.4f} | lr={scheduler.get_last_lr()[0]:.2e}")

        if no_improve >= TRAIN_PARAMS["patience"]:
            print(f"  Early stopping at epoch {epoch}. Best val AUC: {best_val_auc:.4f}")
            break

    # ── load best weights
    model.load_state_dict(best_state)
    model.eval()

    # ── temperature calibration on val set
    Xv = torch.tensor(inp.X_val.values.astype(np.float32)).to(DEVICE)
    yv = torch.tensor(inp.y_val.values.astype(np.float32))
    with torch.no_grad():
        p24v, p72v = model(Xv)
        logits_v = (p24v if target == "y_24h" else p72v).cpu()
        # Convert probabilities back to logits for calibration
        logits_v = torch.logit(logits_v.clamp(1e-6, 1 - 1e-6))

    scaler = TemperatureScaler()
    scaler.fit(logits_v, yv)
    print(f"  Temperature: {scaler.temperature.item():.4f}")

    trained_models[target] = {"model": model, "scaler": scaler}


===== Training y_24h =====
  Epoch   1 | val_auc=0.7710 | best=0.7710 | lr=1.00e-04
  Epoch  25 | val_auc=0.7989 | best=0.8052 | lr=9.62e-05
  Epoch  50 | val_auc=0.7985 | best=0.8134 | lr=8.55e-05
  Early stopping at epoch 57. Best val AUC: 0.8134
  Temperature: 1.1240

===== Training y_72h =====
  Epoch   1 | val_auc=0.7431 | best=0.7431 | lr=1.00e-04
  Epoch  25 | val_auc=0.7543 | best=0.7670 | lr=9.62e-05
  Epoch  50 | val_auc=0.7640 | best=0.7697 | lr=8.55e-05
  Epoch  75 | val_auc=0.7705 | best=0.7717 | lr=6.94e-05
  Epoch 100 | val_auc=0.7686 | best=0.7729 | lr=5.05e-05
  Early stopping at epoch 114. Best val AUC: 0.7729
  Temperature: 1.5150


In [9]:
summary_rows = []

for target in TARGETS:
    inp     = prepared[target]
    model   = trained_models[target]["model"]
    scaler  = trained_models[target]["scaler"]
    horizon = int(target.split("_")[1].replace("h", ""))

    for split_name, X_df, y_s in [
        ("train", inp.X_train, inp.y_train),
        ("val",   inp.X_val,   inp.y_val),
        ("test",  inp.X_test,  inp.y_test),
    ]:
        X_t = torch.tensor(X_df.values.astype(np.float32)).to(DEVICE)
        with torch.no_grad():
            p24, p72 = model(X_t)
            raw_prob = (p24 if target == "y_24h" else p72).cpu()
            logits   = torch.logit(raw_prob.clamp(1e-6, 1 - 1e-6))
            cal_prob = scaler(logits).detach().numpy()

        for calibrated, y_prob in [(False, raw_prob.numpy()), (True, cal_prob)]:
            m = evaluate_binary_probabilities(y_true=y_s, y_prob=y_prob)
            suffix = "" if calibrated else "_uncalibrated"
            summary_rows.append({
                "model_name": MODEL_NAME + suffix,
                "split":      split_name,
                "horizon":    horizon,
                "calibrated": calibrated,
                **m,
            })

_split_order = {"train": 0, "val": 1, "test": 2}
metrics_df = (
    pd.DataFrame(summary_rows)
    .assign(_o=lambda d: d["split"].map(_split_order))
    .sort_values(["model_name", "horizon", "_o"])
    .drop(columns="_o")
    .reset_index(drop=True)
)

# Show calibrated results only
print(metrics_df[metrics_df["calibrated"] == True][
    ["model_name","horizon","split","roc_auc","log_loss","brier_score"]
].to_string(index=False))

model_name  horizon split  roc_auc  log_loss  brier_score
    ftt_v1       24 train 0.841016  0.488399     0.160607
    ftt_v1       24   val 0.813436  0.526708     0.176514
    ftt_v1       24  test 0.806457  0.535482     0.179787
    ftt_v1       72 train 0.852010  0.490258     0.160339
    ftt_v1       72   val 0.772863  0.573915     0.195594
    ftt_v1       72  test 0.753570  0.595552     0.204575


In [10]:
frames = [metrics_df]
for fname in ["mlp_best_v1_metrics.csv", "xgb_best_v4_metrics.csv",
              "baselines_metrics.csv"]:
    p = METRICS_DIR / fname
    if p.exists():
        frames.append(pd.read_csv(p))

comparison = pd.concat(frames, ignore_index=True)
# Keep calibrated row only where the field exists
if "calibrated" in comparison.columns:
    comparison = comparison[comparison["calibrated"].isna() | comparison["calibrated"]]

pivot = (
    comparison
    .pivot_table(index=["horizon","split"], columns="model_name", values="roc_auc")
    .round(4)
)
print("ROC-AUC comparison (all models):")
print(pivot.to_string())

ROC-AUC comparison (all models):
model_name     climatology  ftt_v1  mlp_best_v1  simplified_rj
horizon split                                                 
24      test           0.5  0.8065       0.8069         0.5676
        train          0.5  0.8410       0.8657         0.5816
        val            0.5  0.8134       0.8235         0.5753
72      test           0.5  0.7536       0.7740         0.5652
        train          0.5  0.8520       0.8487         0.5771
        val            0.5  0.7729       0.7961         0.5739


In [11]:
prediction_rows = []

for target in TARGETS:
    inp     = prepared[target]
    model   = trained_models[target]["model"]
    scaler  = trained_models[target]["scaler"]
    horizon = int(target.split("_")[1].replace("h", ""))

    for split_name, X_df, y_s, raw_df in [
        ("train", inp.X_train, inp.y_train, splits_eng["train"]),
        ("val",   inp.X_val,   inp.y_val,   splits_eng["val"]),
        ("test",  inp.X_test,  inp.y_test,  splits_eng["test"]),
    ]:
        X_t = torch.tensor(X_df.values.astype(np.float32)).to(DEVICE)
        with torch.no_grad():
            p24, p72 = model(X_t)
            raw_prob = (p24 if target == "y_24h" else p72).cpu()
            logits   = torch.logit(raw_prob.clamp(1e-6, 1 - 1e-6))
            y_prob   = scaler(logits).detach().numpy()

        prediction_rows.append(pd.DataFrame({
            "trigger_event_id": raw_df.loc[X_df.index, "trigger_event_id"].values,
            "split":            split_name,
            "horizon":          horizon,
            "model_name":       MODEL_NAME,
            "y_true":           y_s.values,
            "y_prob":           y_prob,
        }))

predictions_df = pd.concat(prediction_rows, ignore_index=True)

METRICS_DIR.mkdir(parents=True, exist_ok=True)
predictions_df.to_csv(METRICS_DIR / f"{MODEL_NAME}_predictions.csv", index=False)
metrics_df.to_csv(    METRICS_DIR / f"{MODEL_NAME}_metrics.csv",     index=False)

print(f"Predictions → {METRICS_DIR / f'{MODEL_NAME}_predictions.csv'}")
print(f"Metrics     → {METRICS_DIR / f'{MODEL_NAME}_metrics.csv'}")
print(f"Total rows  : {len(predictions_df)}")

Predictions → /Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10/reports/metrics/ftt_v1_predictions.csv
Metrics     → /Users/tonieenriquez/Desktop/SUTD/SoftwareConstruction/CDS_Group10/reports/metrics/ftt_v1_metrics.csv
Total rows  : 56576
